# Washington RGB-D Object Dataset Preprocessing Pipeline

Preprocesses the [Washington RGB-D Object Dataset](https://rgbd-dataset.cs.washington.edu/dataset/rgbd-dataset_eval/) eval set
for use in the MSNN dual-stream (RGB + Depth) pipeline.

## Dataset Facts
- **300 objects** across **51 categories** (e.g. apple, banana, bowl, camera, ...)
- Each object has **3 video sequences** captured at different turntable elevation angles (~30°, 45°, 60°)
- Eval set contains **cropped bounding-box frames** subsampled at every 5th video frame
- **RGB:** uint8, 3-channel crop (`_crop.png`) — variable size bounding boxes
- **Depth:** uint16, single-channel (`_depthcrop.png`) — values in **millimeters**, 0 = missing/invalid
- **Mask:** binary segmentation (`_maskcrop.png`) — object vs background
- **Naming:** `<category>_<instance>_<video>_<frame>_[crop|depthcrop|maskcrop].png`

## Pipeline Steps
1. Extract zip & explore dataset structure
2. Stratified, sequence-aware downsampling to ~15K frames
3. Pad to square (preserve aspect ratio) → resize to 256×256 → apply mask
4. Save as paired `_rgb.pt` / `_depth.pt` tensors
5. Object-level 70/15/15 train/val/test split (no leakage)
6. Compute normalization statistics (train only, depth in meters, zeros excluded)
7. Package as `tar.gz` to Google Drive

**Runtime:** Colab (High RAM recommended), no GPU needed.

## 1. Imports & Dependencies

In [ ]:
import glob
import json
import math
import os
import random
import re
import shutil
import subprocess
from collections import Counter, defaultdict
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
from tqdm.auto import tqdm

## 2. Configuration

In [ ]:
ZIP_PATH      = '/content/drive/Shareddrives/MSNN-Capstone/data/rgbd-dataset_eval.zip'
EXTRACT_DIR   = '/content/rgbd-dataset_eval'
BASE_OUT_DIR  = '/content/washington_rgbd_256'
DRIVE_OUT     = '/content/drive/Shareddrives/MSNN-Capstone/data'

TARGET_SAMPLES = 15000
TARGET_SIZE    = 256

TRAIN_FRAC = 0.70
VAL_FRAC   = 0.15
TEST_FRAC  = 0.15

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

## 3. Mount Drive & Extract

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

if not os.path.exists(EXTRACT_DIR):
    print(f'Extracting {ZIP_PATH} ...')
    subprocess.run(['unzip', '-q', ZIP_PATH, '-d', '/content'], check=True)
    print('Done.')
else:
    print(f'{EXTRACT_DIR} already exists, skipping extraction.')

# The zip may extract to a subfolder; find the actual root
candidate_roots = [
    EXTRACT_DIR,
    os.path.join('/content', 'rgbd-dataset'),
    os.path.join('/content', 'rgbd-dataset_eval'),
]
DATASET_ROOT = None
for root in candidate_roots:
    if os.path.isdir(root):
        subdirs = [d for d in os.listdir(root)
                   if os.path.isdir(os.path.join(root, d))]
        if len(subdirs) > 10:  # expect 51 category folders
            DATASET_ROOT = root
            break

if DATASET_ROOT is None:
    # Try one level deeper
    for root in candidate_roots:
        if os.path.isdir(root):
            for sub in os.listdir(root):
                sub_path = os.path.join(root, sub)
                if os.path.isdir(sub_path):
                    inner = [d for d in os.listdir(sub_path)
                             if os.path.isdir(os.path.join(sub_path, d))]
                    if len(inner) > 10:
                        DATASET_ROOT = sub_path
                        break
            if DATASET_ROOT:
                break

assert DATASET_ROOT is not None, (
    'Could not find dataset root with category folders. '
    'Check extraction path and zip structure.'
)
print(f'Dataset root: {DATASET_ROOT}')
print(f'Category folders: {len([d for d in os.listdir(DATASET_ROOT) if os.path.isdir(os.path.join(DATASET_ROOT, d))])}')

## 4. Exploratory Analysis

Scan the dataset to discover categories, objects, video sequences, frame
counts, and crop sizes. **Review the output** to verify assumptions before
proceeding.

In [ ]:
# ---- Discover all frames ----
# Naming: <category>_<instance>_<video>_<frame>_crop.png
# We parse from the filename, not from directory structure, for robustness.

all_crop_files = sorted(glob.glob(os.path.join(DATASET_ROOT, '*', '*', '*_crop.png')))
print(f'Total _crop.png files found: {len(all_crop_files)}')

# Parse each filename into (category, instance, video, frame)
# The category may contain underscores (e.g. bell_pepper, cell_phone),
# so we parse from the instance number backwards.
# Format: <category>_<instance_num>_<video_num>_<frame_num>_crop.png
# Instance num, video num, frame num are all integers.


frame_records = []  # list of dicts
parse_errors = []

for fpath in all_crop_files:
    fname = os.path.basename(fpath)
    # Remove _crop.png suffix
    stem = fname.replace('_crop.png', '')
    # Split from the right: last part is frame, then video, then instance num
    parts = stem.rsplit('_', 3)
    if len(parts) < 4:
        parse_errors.append(fname)
        continue
    # parts = [category_maybe_with_underscores, instance_num, video_num, frame_num]
    try:
        frame_num = int(parts[-1])
        video_num = int(parts[-2])
        instance_num = int(parts[-3])
        category = parts[-4]
    except (ValueError, IndexError):
        parse_errors.append(fname)
        continue

    object_id = f'{category}_{instance_num}'  # unique object identifier
    frame_records.append({
        'path': fpath,
        'category': category,
        'instance_num': instance_num,
        'object_id': object_id,
        'video_num': video_num,
        'frame_num': frame_num,
        'filename': fname,
    })

if parse_errors:
    print(f'WARNING: {len(parse_errors)} files could not be parsed:')
    for e in parse_errors[:10]:
        print(f'  {e}')

print(f'Successfully parsed: {len(frame_records)} frames')

# ---- Aggregate statistics ----
categories = sorted(set(r['category'] for r in frame_records))
objects = sorted(set(r['object_id'] for r in frame_records))

# Frames per object, broken down by video sequence
obj_video_frames = defaultdict(lambda: defaultdict(list))
for r in frame_records:
    obj_video_frames[r['object_id']][r['video_num']].append(r)

# Category -> list of object IDs
cat_objects = defaultdict(set)
for r in frame_records:
    cat_objects[r['category']].add(r['object_id'])

print(f'\n{"="*70}')
print(f'Categories: {len(categories)}')
print(f'Objects:    {len(objects)}')
print(f'Frames:     {len(frame_records)}')
print(f'{"="*70}')

# Per-category summary
print(f'\n{"Category":<25} {"Objects":>7} {"Frames":>7} {"Frames/Obj":>10} {"Videos":>7}')
print('-' * 60)
for cat in categories:
    cat_obj_ids = sorted(cat_objects[cat])
    n_objects = len(cat_obj_ids)
    n_frames = sum(1 for r in frame_records if r['category'] == cat)
    videos_per_obj = [len(obj_video_frames[oid]) for oid in cat_obj_ids]
    avg_videos = np.mean(videos_per_obj) if videos_per_obj else 0
    print(f'{cat:<25} {n_objects:>7} {n_frames:>7} {n_frames/n_objects:>10.1f} {avg_videos:>7.1f}')

# Frame count distribution per object
frames_per_obj = [sum(len(v) for v in obj_video_frames[oid].values()) for oid in objects]
if frames_per_obj:
    print(f'\nFrames per object: min={min(frames_per_obj)}, max={max(frames_per_obj)}, '
          f'mean={np.mean(frames_per_obj):.1f}, median={np.median(frames_per_obj):.1f}')
else:
    print('\nNo frames found! Check DATASET_ROOT and glob pattern.')

# Video sequence IDs observed
all_video_nums = sorted(set(r['video_num'] for r in frame_records))
print(f'Video sequence IDs observed: {all_video_nums}')

# Sample a few images to check crop sizes
print(f'\nSample crop sizes (first 20 objects):')
sizes = []
for r in frame_records[:200]:
    img = cv2.imread(r['path'])
    if img is not None:
        sizes.append((img.shape[1], img.shape[0]))  # (w, h)
widths, heights = zip(*sizes)
print(f'  Width:  min={min(widths)}, max={max(widths)}, mean={np.mean(widths):.0f}')
print(f'  Height: min={min(heights)}, max={max(heights)}, mean={np.mean(heights):.0f}')


## 5. Stratified, Sequence-Aware Downsampling

Each object has 3 video sequences at different elevation angles. We divide
each object's frame budget equally across its video sequences, then sample
at even intervals within each sequence. This ensures all elevation angles
are represented and avoids selecting consecutive (similar) frames.

In [ ]:
# ---- Compute per-object frame budget ----
num_objects = len(objects)
base_budget = TARGET_SAMPLES // num_objects
remainder = TARGET_SAMPLES - base_budget * num_objects

print(f'Target samples: {TARGET_SAMPLES}')
print(f'Objects: {num_objects}')
print(f'Base budget per object: {base_budget}')
print(f'Remainder to distribute: {remainder}')

# Give each object a base budget, distribute remainder round-robin
object_budgets = {}
for i, oid in enumerate(sorted(objects)):
    object_budgets[oid] = base_budget + (1 if i < remainder else 0)

# ---- Sequence-aware even-interval sampling ----
selected_frames = []  # list of frame record dicts
sampling_stats = {'took_all': 0, 'sampled': 0, 'surplus': 0}

for oid in sorted(objects):
    budget = object_budgets[oid]
    video_seqs = obj_video_frames[oid]  # {video_num: [records]}
    n_videos = len(video_seqs)

    if n_videos == 0:
        continue

    # Divide budget equally across video sequences
    per_video_budget = budget // n_videos
    video_remainder = budget - per_video_budget * n_videos

    for vi, (vnum, vframes) in enumerate(sorted(video_seqs.items())):
        # Sort frames by frame number
        vframes_sorted = sorted(vframes, key=lambda r: r['frame_num'])
        n_available = len(vframes_sorted)
        v_budget = per_video_budget + (1 if vi < video_remainder else 0)

        if n_available <= v_budget:
            # Take all frames from this video
            selected_frames.extend(vframes_sorted)
            sampling_stats['took_all'] += 1
            sampling_stats['surplus'] += v_budget - n_available
        else:
            # Even-interval sampling
            step = n_available / v_budget
            indices = [int(round(i * step)) for i in range(v_budget)]
            # Clamp to valid range
            indices = [min(idx, n_available - 1) for idx in indices]
            # Remove duplicates while preserving order
            seen = set()
            unique_indices = []
            for idx in indices:
                if idx not in seen:
                    seen.add(idx)
                    unique_indices.append(idx)
            selected_frames.extend(vframes_sorted[i] for i in unique_indices)
            sampling_stats['sampled'] += 1

print(f'\nSelected {len(selected_frames)} frames')
print(f'Video sequences where all frames taken: {sampling_stats["took_all"]}')
print(f'Video sequences downsampled: {sampling_stats["sampled"]}')
print(f'Unused surplus budget: {sampling_stats["surplus"]}')

# Verify per-category distribution
sel_cat_counts = Counter(r['category'] for r in selected_frames)
print(f'\nSelected frames per category:')
for cat in categories:
    print(f'  {cat:<25} {sel_cat_counts.get(cat, 0):>6}')

## 6. Pad, Resize, and Mask

**Order of operations** (critical to avoid artifacts):
1. Pad raw images to square (symmetric zero-padding) — preserves aspect ratio
2. Resize to 256×256 — `INTER_AREA`/`INTER_LINEAR` for RGB, `INTER_NEAREST` for depth & mask
3. Apply resized mask last — avoids interpolation halos at object edges

In [ ]:
def pad_to_square(img):
    """Pad image to square with symmetric zero-padding."""
    h, w = img.shape[:2]
    size = max(h, w)
    pad_top = (size - h) // 2
    pad_bottom = size - h - pad_top
    pad_left = (size - w) // 2
    pad_right = size - w - pad_left
    if img.ndim == 3:
        padded = cv2.copyMakeBorder(img, pad_top, pad_bottom, pad_left, pad_right,
                                    cv2.BORDER_CONSTANT, value=(0, 0, 0))
    else:
        padded = cv2.copyMakeBorder(img, pad_top, pad_bottom, pad_left, pad_right,
                                    cv2.BORDER_CONSTANT, value=0)
    return padded


def process_frame(record, out_dir, target_size):
    """Load, pad, resize, mask, and save a single frame as paired .pt tensors.

    Returns (success: bool, error_msg: str or None).
    """
    fpath = record['path']
    base = os.path.basename(fpath).replace('_crop.png', '')
    category = record['category']

    # Construct sibling file paths
    dir_path = os.path.dirname(fpath)
    rgb_path = fpath
    depth_path = os.path.join(dir_path, f'{base}_depthcrop.png')
    mask_path = os.path.join(dir_path, f'{base}_maskcrop.png')

    # Check all files exist
    for p, name in [(rgb_path, 'RGB'), (depth_path, 'Depth'), (mask_path, 'Mask')]:
        if not os.path.exists(p):
            return False, f'Missing {name}: {p}'

    # Load images
    rgb = cv2.imread(rgb_path, cv2.IMREAD_COLOR)       # BGR uint8
    depth = cv2.imread(depth_path, cv2.IMREAD_UNCHANGED)  # uint16
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)    # uint8

    if rgb is None or depth is None or mask is None:
        return False, f'Failed to read images for {base}'

    # Convert BGR -> RGB
    rgb = cv2.cvtColor(rgb, cv2.COLOR_BGR2RGB)

    # Step 1: Pad to square
    rgb = pad_to_square(rgb)
    depth = pad_to_square(depth)
    mask = pad_to_square(mask)

    # Step 2: Resize to target_size x target_size
    current_size = rgb.shape[0]  # square now
    if current_size > target_size:
        interp_rgb = cv2.INTER_AREA
    else:
        interp_rgb = cv2.INTER_LINEAR

    rgb = cv2.resize(rgb, (target_size, target_size), interpolation=interp_rgb)
    depth = cv2.resize(depth, (target_size, target_size), interpolation=cv2.INTER_NEAREST)
    mask = cv2.resize(mask, (target_size, target_size), interpolation=cv2.INTER_NEAREST)

    # Step 3: Apply mask AFTER resize
    mask_bool = (mask > 0)
    rgb = rgb * mask_bool[:, :, np.newaxis]
    depth = depth * mask_bool

    # Convert to tensors: RGB [3, H, W] uint8, Depth [1, H, W] uint16
    rgb_tensor = torch.from_numpy(rgb.transpose(2, 0, 1)).to(torch.uint8)
    depth_tensor = torch.from_numpy(depth[np.newaxis, :, :].astype(np.uint16)).to(torch.int16)
    # Note: PyTorch has no uint16 dtype; we store as int16. Values up to 32767mm
    # (~32m) fit safely. The depth sensor range is well within this.

    # Save
    cat_dir = os.path.join(out_dir, category)
    os.makedirs(cat_dir, exist_ok=True)

    out_stem = f'{base}_rgb.pt'
    out_depth_stem = f'{base}_depth.pt'
    torch.save(rgb_tensor, os.path.join(cat_dir, out_stem))
    torch.save(depth_tensor, os.path.join(cat_dir, out_depth_stem))

    return True, None


# ---- Process all selected frames ----
STAGING_DIR = os.path.join(BASE_OUT_DIR, '_staging')
os.makedirs(STAGING_DIR, exist_ok=True)

errors = []
for record in tqdm(selected_frames, desc='Processing frames'):
    success, err = process_frame(record, STAGING_DIR, TARGET_SIZE)
    if not success:
        errors.append(err)

processed = len(selected_frames) - len(errors)
print(f'\nProcessed: {processed}')
if errors:
    print(f'Errors: {len(errors)}')
    for e in errors[:10]:
        print(f'  {e}')

## 7. Verify Processed Tensors

In [ ]:
corrupt_files = []
checked = 0
unpaired = []

for cls in sorted(os.listdir(STAGING_DIR)):
    cls_dir = os.path.join(STAGING_DIR, cls)
    if not os.path.isdir(cls_dir):
        continue

    rgb_files = sorted(glob.glob(os.path.join(cls_dir, '*_rgb.pt')))
    for rgb_path in rgb_files:
        depth_path = rgb_path.replace('_rgb.pt', '_depth.pt')
        if not os.path.exists(depth_path):
            unpaired.append(rgb_path)
            continue

        try:
            rgb_t = torch.load(rgb_path, weights_only=True)
            depth_t = torch.load(depth_path, weights_only=True)

            assert rgb_t.shape == (3, TARGET_SIZE, TARGET_SIZE), f'RGB shape {rgb_t.shape}'
            assert rgb_t.dtype == torch.uint8, f'RGB dtype {rgb_t.dtype}'
            assert depth_t.shape == (1, TARGET_SIZE, TARGET_SIZE), f'Depth shape {depth_t.shape}'
            assert depth_t.dtype == torch.int16, f'Depth dtype {depth_t.dtype}'
            checked += 1
        except Exception as e:
            corrupt_files.append((rgb_path, str(e)))

print(f'Checked: {checked} pairs')
if unpaired:
    print(f'Unpaired RGB files (missing depth): {len(unpaired)}')
if corrupt_files:
    print(f'Corrupt files: {len(corrupt_files)}')
    for path, err in corrupt_files[:5]:
        print(f'  {path}: {err}')
    # Delete corrupt
    for path, _ in corrupt_files:
        os.remove(path)
        depth_p = path.replace('_rgb.pt', '_depth.pt')
        if os.path.exists(depth_p):
            os.remove(depth_p)
    print(f'Deleted {len(corrupt_files)} corrupt pairs.')
else:
    print('All tensors valid.')

## 8. Object-Level Train/Val/Test Split

Split **objects** (not frames) to prevent data leakage. Handles small
categories with fallback logic:
- \>=4 objects: standard 70/15/15
- 3 objects: 1 train, 1 val, 1 test
- 2 objects: 1 train, 1 val
- 1 object: train only

In [ ]:
random.seed(SEED)

# Build object -> category mapping from staged files
staged_objects = defaultdict(set)  # category -> set of object_ids
for cls in os.listdir(STAGING_DIR):
    cls_dir = os.path.join(STAGING_DIR, cls)
    if not os.path.isdir(cls_dir):
        continue
    for f in os.listdir(cls_dir):
        if f.endswith('_rgb.pt'):
            # Parse object_id: <category>_<instance>_<video>_<frame>_rgb.pt
            stem = f.replace('_rgb.pt', '')
            parts = stem.rsplit('_', 2)  # [category_instance, video, frame]
            if len(parts) >= 3:
                obj_id = parts[0]  # category_instance
                staged_objects[cls].add(obj_id)

# Perform split
split_assignment = {}  # object_id -> 'train'/'val'/'test'
split_warnings = []

for cat in sorted(staged_objects.keys()):
    obj_ids = sorted(staged_objects[cat])
    random.shuffle(obj_ids)
    n = len(obj_ids)

    if n >= 4:
        n_test = max(1, round(n * TEST_FRAC))
        n_val = max(1, round(n * VAL_FRAC))
        n_train = n - n_val - n_test
        if n_train < 1:
            n_train = 1
            n_val = max(1, (n - 1) // 2)
            n_test = n - 1 - n_val
    elif n == 3:
        n_train, n_val, n_test = 1, 1, 1
    elif n == 2:
        n_train, n_val, n_test = 1, 1, 0
        split_warnings.append(f'{cat}: only 2 objects, no test split')
    else:  # n == 1
        n_train, n_val, n_test = 1, 0, 0
        split_warnings.append(f'{cat}: only 1 object, train only')

    for oid in obj_ids[:n_train]:
        split_assignment[oid] = 'train'
    for oid in obj_ids[n_train:n_train + n_val]:
        split_assignment[oid] = 'val'
    for oid in obj_ids[n_train + n_val:n_train + n_val + n_test]:
        split_assignment[oid] = 'test'

if split_warnings:
    print('Split warnings:')
    for w in split_warnings:
        print(f'  {w}')

# ---- Move files into split directories ----
for split in ['train', 'val', 'test']:
    os.makedirs(os.path.join(BASE_OUT_DIR, split), exist_ok=True)

moved_counts = Counter()

for cls in sorted(os.listdir(STAGING_DIR)):
    cls_dir = os.path.join(STAGING_DIR, cls)
    if not os.path.isdir(cls_dir):
        continue

    for f in sorted(os.listdir(cls_dir)):
        if not f.endswith('.pt'):
            continue
        # Determine object_id
        stem = f.replace('_rgb.pt', '').replace('_depth.pt', '')
        parts = stem.rsplit('_', 2)
        if len(parts) >= 3:
            obj_id = parts[0]
        else:
            continue

        assigned_split = split_assignment.get(obj_id)
        if assigned_split is None:
            continue

        dest_dir = os.path.join(BASE_OUT_DIR, assigned_split, cls)
        os.makedirs(dest_dir, exist_ok=True)
        shutil.move(os.path.join(cls_dir, f), os.path.join(dest_dir, f))
        if f.endswith('_rgb.pt'):
            moved_counts[assigned_split] += 1

# Clean up staging
shutil.rmtree(STAGING_DIR, ignore_errors=True)

# ---- Print summary ----
print(f'\n{"Split":<8} {"Objects":>8} {"Frames":>8} {"Pct":>6}')
print('-' * 35)
total_frames = sum(moved_counts.values())
for split in ['train', 'val', 'test']:
    n_obj = sum(1 for v in split_assignment.values() if v == split)
    n_frames = moved_counts[split]
    pct = 100.0 * n_frames / total_frames if total_frames > 0 else 0
    print(f'{split:<8} {n_obj:>8} {n_frames:>8} {pct:>5.1f}%')
print(f'{"TOTAL":<8} {len(split_assignment):>8} {total_frames:>8}')

# Per-category breakdown
print(f'\n{"Category":<25} {"Train":>6} {"Val":>6} {"Test":>6}')
print('-' * 50)
for cat in sorted(staged_objects.keys()):
    counts = {s: 0 for s in ['train', 'val', 'test']}
    for oid in staged_objects[cat]:
        s = split_assignment.get(oid)
        if s:
            counts[s] += 1
    print(f'{cat:<25} {counts["train"]:>6} {counts["val"]:>6} {counts["test"]:>6}')


## 9. Data Leakage Test

Verify no object appears in more than one split.

In [ ]:
split_objects = defaultdict(set)

for split in ['train', 'val', 'test']:
    split_dir = os.path.join(BASE_OUT_DIR, split)
    if not os.path.exists(split_dir):
        continue
    for cls in os.listdir(split_dir):
        cls_dir = os.path.join(split_dir, cls)
        if not os.path.isdir(cls_dir):
            continue
        for f in os.listdir(cls_dir):
            if f.endswith('_rgb.pt'):
                stem = f.replace('_rgb.pt', '')
                parts = stem.rsplit('_', 2)
                if len(parts) >= 3:
                    split_objects[split].add(parts[0])

# Check pairwise intersections
leaks_found = False
for s1, s2 in [('train', 'val'), ('train', 'test'), ('val', 'test')]:
    overlap = split_objects[s1] & split_objects[s2]
    if overlap:
        print(f'LEAKAGE: {len(overlap)} objects in both {s1} and {s2}: {overlap}')
        leaks_found = True

if not leaks_found:
    print('No data leakage detected.')
    for split in ['train', 'val', 'test']:
        print(f'  {split}: {len(split_objects[split])} unique objects')

## 10. Frame Diversity Verification

Verify even-interval sampling produced visually diverse frames by computing
pixel differences between consecutive selected frames within each video
sequence.

In [ ]:
random.seed(SEED)
DIVERSITY_SAMPLE = 10

train_objects = sorted(split_objects.get('train', set()))
sample_objects = random.sample(train_objects, min(DIVERSITY_SAMPLE, len(train_objects)))

print('Frame diversity check (mean absolute pixel difference between consecutive frames):')
print(f'Checking {len(sample_objects)} objects...\n')

for obj_id in sample_objects:
    # Find category from obj_id
    parts = obj_id.rsplit('_', 1)
    cat = parts[0] if len(parts) == 2 else obj_id

    # Find all RGB files for this object in train
    obj_files = sorted(glob.glob(
        os.path.join(BASE_OUT_DIR, 'train', cat, f'{obj_id}_*_rgb.pt')
    ))
    if len(obj_files) < 2:
        # Try with the category matching from directory
        for cat_dir in glob.glob(os.path.join(BASE_OUT_DIR, 'train', '*')):
            found = sorted(glob.glob(os.path.join(cat_dir, f'{obj_id}_*_rgb.pt')))
            if found:
                obj_files = found
                break

    if len(obj_files) < 2:
        print(f'  {obj_id}: only {len(obj_files)} frames, skipping')
        continue

    diffs = []
    prev = torch.load(obj_files[0], weights_only=True).float()
    for fpath in obj_files[1:]:
        curr = torch.load(fpath, weights_only=True).float()
        diff = (curr - prev).abs().mean().item()
        diffs.append(diff)
        prev = curr

    min_d = min(diffs)
    mean_d = np.mean(diffs)
    flag = ' <-- LOW' if min_d < 3.0 else ''
    print(f'  {obj_id}: {len(obj_files)} frames, min_diff={min_d:.1f}, mean_diff={mean_d:.1f}{flag}')

## 11. Dataset Statistics

In [ ]:
for split in ['train', 'val', 'test']:
    split_dir = os.path.join(BASE_OUT_DIR, split)
    if not os.path.exists(split_dir):
        print(f'\n=== {split.upper()} === (not found)')
        continue
    print(f'\n=== {split.upper()} ===')
    total = 0
    class_counts = {}
    for cls in sorted(os.listdir(split_dir)):
        cls_dir = os.path.join(split_dir, cls)
        if not os.path.isdir(cls_dir):
            continue
        n = len(glob.glob(os.path.join(cls_dir, '*_rgb.pt')))
        class_counts[cls] = n
        total += n
    for cls, n in sorted(class_counts.items()):
        print(f'  {cls:<25} {n:>6}')
    print(f'  {"TOTAL":<25} {total:>6}')

# Depth range stats from a sample
print('\n--- Depth Stats (sample of 200 train frames) ---')
train_depth_files = sorted(glob.glob(
    os.path.join(BASE_OUT_DIR, 'train', '*', '*_depth.pt')
))[:200]

depth_mins, depth_maxs, zero_fracs = [], [], []
for dp in train_depth_files:
    d = torch.load(dp, weights_only=True).float()
    mask_valid = d > 0
    if mask_valid.any():
        depth_mins.append(d[mask_valid].min().item())
        depth_maxs.append(d[mask_valid].max().item())
    zero_fracs.append((d == 0).float().mean().item())

print(f'  Non-zero depth min (mm): min={min(depth_mins):.0f}, max={max(depth_mins):.0f}, mean={np.mean(depth_mins):.0f}')
print(f'  Non-zero depth max (mm): min={min(depth_maxs):.0f}, max={max(depth_maxs):.0f}, mean={np.mean(depth_maxs):.0f}')
print(f'  Zero fraction: min={min(zero_fracs):.3f}, max={max(zero_fracs):.3f}, mean={np.mean(zero_fracs):.3f}')

## 12. Normalization Statistics

Streaming Welford computation from **train split only**.
- RGB normalized to \[0, 1\], per-channel mean/std
- Depth converted to meters (/ 1000), **all zero-valued pixels excluded**
  (both masked background AND sensor-invalid readings)

In [ ]:
print('Computing normalization statistics from train split...')

all_rgb_paths = sorted(glob.glob(
    os.path.join(BASE_OUT_DIR, 'train', '*', '*_rgb.pt')
))
all_depth_paths = [p.replace('_rgb.pt', '_depth.pt') for p in all_rgb_paths]

print(f'Train samples: {len(all_rgb_paths)}')

# Batch Welford online mean/variance (per-channel for RGB)
rgb_n = np.zeros(3, dtype=np.int64)
rgb_mean = np.zeros(3, dtype=np.float64)
rgb_m2 = np.zeros(3, dtype=np.float64)

depth_n = 0
depth_mean = np.float64(0.0)
depth_m2 = np.float64(0.0)

for rgb_path, depth_path in tqdm(zip(all_rgb_paths, all_depth_paths),
                                  total=len(all_rgb_paths),
                                  desc='Normalization stats'):
    # RGB: [3, 256, 256] uint8 -> float64 in [0, 1]
    rgb = torch.load(rgb_path, weights_only=True).numpy().astype(np.float64) / 255.0
    for c in range(3):
        pixels = rgb[c].ravel()
        batch_n = len(pixels)
        batch_mean = pixels.mean()
        batch_var = pixels.var() * batch_n  # sum of squared deviations

        new_n = rgb_n[c] + batch_n
        delta = batch_mean - rgb_mean[c]
        rgb_mean[c] = (rgb_n[c] * rgb_mean[c] + batch_n * batch_mean) / new_n
        rgb_m2[c] += batch_var + delta ** 2 * rgb_n[c] * batch_n / new_n
        rgb_n[c] = new_n

    # Depth: [1, 256, 256] int16 -> float64 meters, exclude zeros
    depth = torch.load(depth_path, weights_only=True).numpy().astype(np.float64)
    depth_meters = depth[0] / 1000.0  # mm -> meters
    valid = depth_meters[depth_meters > 0].ravel()
    if len(valid) > 0:
        batch_n = len(valid)
        batch_mean = valid.mean()
        batch_var = valid.var() * batch_n

        new_n = depth_n + batch_n
        delta = batch_mean - depth_mean
        depth_mean = (depth_n * depth_mean + batch_n * batch_mean) / new_n
        depth_m2 += batch_var + delta ** 2 * depth_n * batch_n / new_n
        depth_n = new_n

rgb_std = np.sqrt(rgb_m2 / (rgb_n - 1))
depth_std = math.sqrt(depth_m2 / (depth_n - 1)) if depth_n > 1 else 0.0

norm_stats = {
    'rgb_mean': rgb_mean.tolist(),
    'rgb_std': rgb_std.tolist(),
    'depth_mean': float(depth_mean),
    'depth_std': float(depth_std),
    'depth_unit': 'meters',
    'rgb_range': '[0, 1]',
    'computed_from': 'train split only',
    'depth_zero_excluded': True,
}

print(f'\nRGB mean:  {rgb_mean}')
print(f'RGB std:   {rgb_std}')
print(f'Depth mean: {depth_mean:.4f} m')
print(f'Depth std:  {depth_std:.4f} m')
print(f'Depth pixels used: {depth_n:,}')

## 13. Write Metadata Files

In [ ]:
# class_names.txt
# Use categories discovered during exploration
all_categories = sorted(set(
    cls for split in ['train', 'val', 'test']
    if os.path.exists(os.path.join(BASE_OUT_DIR, split))
    for cls in os.listdir(os.path.join(BASE_OUT_DIR, split))
    if os.path.isdir(os.path.join(BASE_OUT_DIR, split, cls))
))

class_names_path = os.path.join(BASE_OUT_DIR, 'class_names.txt')
with open(class_names_path, 'w') as f:
    for name in all_categories:
        f.write(f'{name}\n')
print(f'Wrote {class_names_path} ({len(all_categories)} classes)')

# norm_stats.json
norm_stats_path = os.path.join(BASE_OUT_DIR, 'norm_stats.json')
with open(norm_stats_path, 'w') as f:
    json.dump(norm_stats, f, indent=2)
print(f'Wrote {norm_stats_path}')

# dataset_info.txt
train_count = len(glob.glob(os.path.join(BASE_OUT_DIR, 'train', '*', '*_rgb.pt')))
val_count = len(glob.glob(os.path.join(BASE_OUT_DIR, 'val', '*', '*_rgb.pt')))
test_count = len(glob.glob(os.path.join(BASE_OUT_DIR, 'test', '*', '*_rgb.pt')))

info_path = os.path.join(BASE_OUT_DIR, 'dataset_info.txt')
with open(info_path, 'w') as f:
    f.write('Washington RGB-D Object Dataset (Eval Set) - Preprocessed\n')
    f.write(f'Classes: {len(all_categories)}\n')
    f.write(f'Train samples: {train_count}\n')
    f.write(f'Val samples:   {val_count}\n')
    f.write(f'Test samples:  {test_count}\n')
    f.write(f'Total samples: {train_count + val_count + test_count}\n')
    f.write(f'\nRGB format:   [3, 256, 256] uint8 (0-255), masked\n')
    f.write(f'Depth format: [1, 256, 256] int16 (millimeters), 0=invalid, masked\n')
    f.write(f'Images padded to square (aspect-ratio preserved) then resized to 256x256\n')
    f.write(f'\nNormalization (train split only):\n')
    f.write(f'  RGB mean:  {norm_stats["rgb_mean"]}\n')
    f.write(f'  RGB std:   {norm_stats["rgb_std"]}\n')
    f.write(f'  Depth mean: {norm_stats["depth_mean"]:.4f} m\n')
    f.write(f'  Depth std:  {norm_stats["depth_std"]:.4f} m\n')
    f.write(f'  Depth zeros excluded from stats\n')
    f.write(f'\nSplit strategy: object-level (no frame leakage across splits)\n')
    f.write(f'Sampling: sequence-aware even-interval (balanced across elevation angles)\n')
print(f'Wrote {info_path}')

## 14. Spot Check Visualization

In [ ]:
random.seed(SEED)

# Show one random sample per category from train split
train_dir = os.path.join(BASE_OUT_DIR, 'train')
train_cats = sorted([d for d in os.listdir(train_dir)
                     if os.path.isdir(os.path.join(train_dir, d))])

n_cats = len(train_cats)
cols = 4  # pairs per row
rows = math.ceil(n_cats / cols)

fig, axes = plt.subplots(rows, cols * 2, figsize=(cols * 6, rows * 3))
if rows == 1:
    axes = axes[np.newaxis, :]

for i, cat in enumerate(train_cats):
    row = i // cols
    col = (i % cols) * 2

    cat_dir = os.path.join(train_dir, cat)
    rgb_files = sorted(glob.glob(os.path.join(cat_dir, '*_rgb.pt')))

    if not rgb_files:
        axes[row, col].set_title(f'{cat} (empty)')
        axes[row, col].axis('off')
        axes[row, col + 1].axis('off')
        continue

    rgb_path = random.choice(rgb_files)
    depth_path = rgb_path.replace('_rgb.pt', '_depth.pt')

    rgb = torch.load(rgb_path, weights_only=True).numpy().transpose(1, 2, 0)
    depth = torch.load(depth_path, weights_only=True).numpy()[0].astype(np.float32)

    axes[row, col].imshow(rgb)
    axes[row, col].set_title(cat, fontsize=8)
    axes[row, col].axis('off')

    axes[row, col + 1].imshow(depth, cmap='viridis')
    axes[row, col + 1].set_title('depth', fontsize=8)
    axes[row, col + 1].axis('off')

# Turn off unused axes
for i in range(n_cats, rows * cols):
    row = i // cols
    col = (i % cols) * 2
    axes[row, col].axis('off')
    axes[row, col + 1].axis('off')

plt.tight_layout()
plt.show()

## 15. Package & Upload to Drive

In [ ]:
local_tar = '/content/washington_rgbd_256.tar.gz'

print('Creating tar.gz archive...')
subprocess.run([
    'tar', 'czf', local_tar,
    '-C', os.path.dirname(BASE_OUT_DIR),
    os.path.basename(BASE_OUT_DIR),
], check=True)

tar_size = os.path.getsize(local_tar) / (1024 ** 3)
print(f'Archive size: {tar_size:.2f} GB')

# Copy to Drive
os.makedirs(DRIVE_OUT, exist_ok=True)
drive_dest = os.path.join(DRIVE_OUT, 'washington_rgbd_256.tar.gz')
print(f'Copying to {drive_dest}...')
shutil.copy2(local_tar, drive_dest)
print(f'Done. Archive at {drive_dest}')